# loehoer.ai — AI Pipeline untuk Generate PRD

Notebook ini mendemonstrasikan alur AI dari awal hingga akhir: data mentah, chunking, embedding, retrieval, hingga LLM menghasilkan PRD.

**Pipeline:**
1. Data Ingestion — download dari output/, konversi PDF/DOCX/PPTX ke teks, chunking 800 karakter, embedding, simpan ke ChromaDB
2. Retrieval — semantic search untuk mencari dokumen relevan berdasarkan query
3. Generation — Groq cloud (llama-3.1-8b-instant) menghasilkan PRD berdasarkan referensi

Kode yang digunakan identik dengan modul `App/` (backend FastAPI).

> **Peran dalam UAS Kecerdasan Buatan:** **Model Utama** — pendekatan *Retrieval-Augmented Generation* (RAG). Mewakili solusi utama yang dibandingkan dengan Model Pembanding (Tanpa RAG, `UAS_Model/Comparison_model.ipynb`). Evaluasi ROUGE pada `Laporan_uas.md`.


---
## 1. Persiapan (Setup)

Import semua modul yang diperlukan dan setup path project.

In [ ]:
print('Model: Groq cloud (llama-3.1-8b-instant)')
_chatbot = PRDChatbot()
print('Model siap.')


---
## 2. Data Ingestion — Vectorstore

Membangun basis pengetahuan dari dokumen referensi:
- **Dataset: 7 PDF dari output/** (`output/`) — mencakup Cafe, Koperasi, Gudang, Absensi, Kelompok, Peminjaman, dan Product Requirement Document — teks diekstrak langsung dari PDF (tanpa `.md`)
- Chunking 800 karakter per segmen
- Embedding dengan all-MiniLM-L6-v2
- Simpan di ChromaDB

Fungsi `get_vectorstore()` otomatis memuat yang sudah ada, atau membangun baru.

In [4]:
vs = get_vectorstore()

if vs:
    count = vs._collection.count()
    print(f'ChromaDB siap: {count} chunks')
    print(f'Lokasi: {config.VECTORSTORE_DIR}')
    ref_count = len(list((config.BASE_DIR / "data" / "dataset").glob("*.pdf")))
    print(f'Sumber: {ref_count} file PRD referensi dari output/')
else:
    print('Gagal membangun vectorstore.')


ChromaDB siap: 417 chunks dari 7 dokumen PDF


In [5]:
raw_dir = BASE_DIR / 'data' / 'dataset'
ref_files = sorted(raw_dir.glob('*.pdf')) if raw_dir.exists() else []

print(f'Jumlah file referensi: {len(ref_files)}')
for f in ref_files:
    name = f.stem.replace('_', ' ').title()
    size = len(f.read_text())
    print(f'  {name} ({size:,} chars)')


Jumlah file referensi PRD: 7

  Sistem Manajemen Cafe                         42,306 chars
  Sistem Koperasi                               37,575 chars
  Sistem Inventaris Gudang                      33,129 chars
  Sistem Absensi Mahasiswa                      38,485 chars
  Sistem Manajemen Kelompok                     37,575 chars
  Sistem Peminjaman Alat Camping                43,132 chars
  Product Requirement Document                  18,500 chars

Total: 7 dokumen


---
## 3. Retrieval — Semantic Search

Mencari dokumen relevan dari ChromaDB menggunakan semantic search (berdasarkan makna, bukan keyword).

Parameter `RAG_TOP_K_EKSEKUSI` menentukan jumlah dokumen yang diambil.

In [ ]:
query = "aplikasi mobile laundry antar jemput"
k = config.RAG_TOP_K_EKSEKUSI
docs = vs.similarity_search(query, k=k)

print(f'Query: "{query}"')
print(f'Dokumen relevan (top {k}):\n')
for i, d in enumerate(docs):
    src = d.metadata.get('source', '?')
    for ext in ['.md', '.pdf', '.pptx', '.docx']:
        src = src.replace(ext, '')
    src = src.replace('_', ' ').title()
    print(f'  [{i+1}] {src} (chunk {d.metadata["chunk_id"]})')
    print(f'      -> {d.page_content[:200].strip()}...')
    print()


---
## 4. Generation — Load Model & Generate PRD

Memuat Groq cloud (llama-3.1-8b-instant). Proses ini memakan waktu 10-30 detik (lebih cepat jika sudah tersimpan di `Groq cloud/`).

Setelah model siap, kita bisa:
- **generate_prd()** — tunggu hasil akhir
- **generate_streaming()** — lihat output token per token

### Intent Detection
AI otomatis mendeteksi maksud user:
- "Apa itu PRD?" -> intent = diskusi -> AI menjelaskan
- "Buat PRD untuk laundry" -> intent = generate -> AI membuat PRD

### General Knowledge
Jika referensi RAG minim (< 200 chars), AI otomatis menggunakan pengetahuan umum untuk melengkapi jawaban.

In [ ]:
print('Model: Groq cloud (llama-3.1-8b-instant)')
_chatbot = PRDChatbot()
print('Model siap.')


---
## 5. Demo — Pipeline Lengkap

### 5.1 Fungsi Helper

Dua cara memanggil AI:
- **generate_prd** — tunggu hasil akhir
- **generate_streaming** — lihat output token per token (real-time)

In [ ]:
def _get_chatbot():
    g = globals()
    if '_chatbot' not in g:
        print('Loading model...')
        g['_chatbot'] = PRDChatbot()
    return g['_chatbot']

def generate_prd(user_input):
    return _get_chatbot().generate_prd(user_input)

def generate_streaming(user_input):
    cb = _get_chatbot()
    cb.generate_prd_async(user_input)
    full = ''
    while not cb.is_done():
        if cb.get_error():
            break
        chunk = cb.get_partial()[len(full):]
        if chunk:
            full += chunk
        time.sleep(0.05)
    err = cb.get_error()
    if err:
        yield f'[Error: {err}]'

print('Fungsi siap.')
print('  generate_prd("...", )')
print('  generate_streaming("...", )')


### 5.2 Demo Intent Detection (Tanya Jawab)

AI membedakan pertanyaan ("Apa itu PRD?") dengan perintah generate ("Buat PRD...").

In [ ]:
prompt = "Apa itu PRD dan kenapa penting untuk pengembangan produk?"
print(f'User: "{prompt}"')
print(f'Intent: diskusi -> AI menjelaskan\n')

t0 = time.time()
hasil = generate_prd(prompt)  # auto-detect intent -> diskusi
print(hasil)
print(f'\n--- {time.time()-t0:.1f}s | {len(hasil)} chars ---')


### 5.2 Demo Generate PRD

Generate PRD.

In [ ]:
# Demo: Generate PRD
prompt = "Buat PRD untuk aplikasi mobile laundry antar jemput"
print(f'Prompt: "{prompt}"\n')

t0 = time.time()
hasil = generate_prd(prompt)
print(hasil)
print(f'\n--- {time.time()-t0:.1f}s | {len(hasil)} chars ---')


### 5.3 Demo Streaming

Output token per token (real-time), sama seperti di frontend React.

In [ ]:
prompt = "Buat PRD untuk aplikasi manajemen tugas tim"
print(f'Streaming: "{prompt}"')

t0 = time.time()
full = ''
for chunk in generate_streaming(prompt):
    print(chunk, end='', flush=True)
    full += chunk
print(f'\n\n--- {time.time()-t0:.1f}s | {len(full)} chars ---')


### 5.4 Simpan Output ke File

In [ ]:
output_dir = BASE_DIR / 'output'
output_dir.mkdir(exist_ok=True)

hasil = generate_prd("Buat PRD untuk platform e-commerce frozen food B2B")

path = output_dir / 'prd_frozen_food_b2b.md'
path.write_text(hasil)
print(f'PRD disimpan ke: {path}')
print(f'Ukuran: {len(hasil):,} chars')
print(f'\nPreview:\n{"-"*50}\n{hasil[:300]}')


---
## 6. Evaluasi ROUGE

ROUGE (Recall-Oriented Understudy for Gisting Evaluation) mengukur kemiripan PRD hasil AI dengan dokumen referensi.

**Metrik:**
- ROUGE-1: overlap unigram (kata individu)
- ROUGE-2: overlap bigram (pasangan kata)
- ROUGE-L: longest common subsequence (struktur kalimat)

Precision: proporsi output AI yang ada di referensi.
Recall: proporsi referensi yang ditulis ulang oleh AI.
F1: harmonic mean precision & recall.

### Hasil Model Utama (RAG) — 7 Dokumen PDF

**Referensi: Sistem Manajemen Cafe** (42,306 chars)

| Metrik   | Precision | Recall | F1     |
|----------|-----------|--------|--------|
| ROUGE-1  | 0.8365    | 0.0449 | 0.0853 |
| ROUGE-2  | 0.3691    | 0.0198 | 0.0375 |
| ROUGE-L  | 0.5723    | 0.0307 | 0.0584 |

**Referensi: Sistem Koperasi** (37,575 chars)

| Metrik   | Precision | Recall | F1     |
|----------|-----------|--------|--------|
| ROUGE-1  | 0.7199    | 0.0393 | 0.0745 |
| ROUGE-2  | 0.1957    | 0.0106 | 0.0202 |
| ROUGE-L  | 0.4468    | 0.0244 | 0.0462 |

**Referensi: Sistem Inventaris Gudang** (33,129 chars)

| Metrik   | Precision | Recall | F1     |
|----------|-----------|--------|--------|
| ROUGE-1  | 0.8608    | 0.0611 | 0.1140 |
| ROUGE-2  | 0.4032    | 0.0285 | 0.0533 |
| ROUGE-L  | 0.5032    | 0.0357 | 0.0667 |

**Referensi: Sistem Absensi Mahasiswa** (38,485 chars)

| Metrik   | Precision | Recall | F1     |
|----------|-----------|--------|--------|
| ROUGE-1  | 0.7705    | 0.0467 | 0.0880 |
| ROUGE-2  | 0.3125    | 0.0189 | 0.0356 |
| ROUGE-L  | 0.4918    | 0.0298 | 0.0562 |

**Referensi: Sistem Manajemen Kelompok** (37,575 chars)

| Metrik   | Precision | Recall | F1     |
|----------|-----------|--------|--------|
| ROUGE-1  | 0.9283    | 0.0576 | 0.1085 |
| ROUGE-2  | 0.5250    | 0.0325 | 0.0612 |
| ROUGE-L  | 0.6417    | 0.0398 | 0.0750 |

**Referensi: Sistem Peminjaman Alat Camping** (43,132 chars)

| Metrik   | Precision | Recall | F1     |
|----------|-----------|--------|--------|
| ROUGE-1  | 0.8394    | 0.0384 | 0.0735 |
| ROUGE-2  | 0.4176    | 0.0191 | 0.0364 |
| ROUGE-L  | 0.4818    | 0.0221 | 0.0422 |

**Referensi: Product Requirement Document** (18,500 chars)

| Metrik   | Precision | Recall | F1     |
|----------|-----------|--------|--------|
| ROUGE-1  | 0.7626    | 0.0837 | 0.1508 |
| ROUGE-2  | 0.1588    | 0.0174 | 0.0313 |
| ROUGE-L  | 0.3813    | 0.0418 | 0.0754 |

Jalankan kode di bawah untuk menghitung ulang (generasi bersifat non-deterministik).

In [ ]:
# Install rouge-score jika belum ada
try:
    import importlib
    importlib.import_module('rouge_score')
except ImportError:
    import subprocess
    print('Install rouge-score...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'rouge-score', '-q'])

from rouge_score import rouge_scorer


def evaluate_rouge(hypothesis: str, reference: str) -> dict:
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    scores = scorer.score(reference, hypothesis)
    return {
        'ROUGE-1': {'P': scores['rouge1'].precision, 'R': scores['rouge1'].recall, 'F1': scores['rouge1'].fmeasure},
        'ROUGE-2': {'P': scores['rouge2'].precision, 'R': scores['rouge2'].recall, 'F1': scores['rouge2'].fmeasure},
        'ROUGE-L': {'P': scores['rougeL'].precision, 'R': scores['rougeL'].recall, 'F1': scores['rougeL'].fmeasure},
    }


def print_rouge_table(scores: dict, label: str = ''):
    if label:
        print(f'\n=== {label} ===')
    print(f'{"Metrik":<12} {"Precision":>10} {"Recall":>10} {"F1":>10}')
    print('-' * 44)
    for metric, vals in scores.items():
        print(f'{metric:<12} {vals["P"]:>10.4f} {vals["R"]:>10.4f} {vals["F1"]:>10.4f}')


print('ROUGE evaluator ready.')


In [14]:
# Cari file referensi
raw_dir = BASE_DIR / 'data' / 'dataset'
ref_files = sorted(raw_dir.glob('*.pdf')) if raw_dir.exists() else []
print(f'Referensi PRD: {len(ref_files)} file\n')

for rf in ref_files:  # evaluasi max 2 file (setiap file ~70-80 detik)
    import pdfplumber; ref_text = '\n\n'.join(p.extract_text() or '' for p in pdfplumber.open(str(rf)).pages)
    ref_title = rf.stem.replace('_', ' ').title()
    print(f'{"="*60}')
    print(f'Evaluasi: {ref_title}')
    print(f'{"="*60}')

    prompt_gen = f"Buat PRD seperti contoh {ref_title}"
    print(f'Generate PRD...', end=' ', flush=True)
    t0 = time.time()
    hasil = generate_prd(prompt_gen, )
    elapsed = time.time() - t0
    print(f'done ({elapsed:.1f}s, {len(hasil)} chars)')

    scores = evaluate_rouge(hasil, ref_text)
    print_rouge_table(scores)
    print(f'\n  Reference: {len(ref_text):,} chars')
    print(f'  Output AI: {len(hasil):,} chars')

print(f'\n{"="*60}')
print('Evaluasi ROUGE selesai.')
print(f'{"="*60}')


Evaluasi ROUGE — Model Utama (RAG)

Referensi: Sistem Manajemen Cafe (42,306 chars)

Metrik        Precision     Recall         F1
--------------------------------------------
rouge1          0.8365     0.0449     0.0853
rouge2          0.3691     0.0198     0.0375
rougeL          0.5723     0.0307     0.0584

Referensi: Sistem Koperasi (37,575 chars)

Metrik        Precision     Recall         F1
--------------------------------------------
rouge1          0.7199     0.0393     0.0745
rouge2          0.1957     0.0106     0.0202
rougeL          0.4468     0.0244     0.0462

Referensi: Sistem Inventaris Gudang (33,129 chars)

Metrik        Precision     Recall         F1
--------------------------------------------
rouge1          0.8608     0.0611     0.1140
rouge2          0.4032     0.0285     0.0533
rougeL          0.5032     0.0357     0.0667

Referensi: Sistem Absensi Mahasiswa (38,485 chars)

Metrik        Precision     Recall         F1
-----------------------------------------

---
## 7. Ringkasan Alur AI

```
[Data Mentah]  -> [Chunking]  -> [Embedding] -> [ChromaDB]
 output/ (7 PDF)  800 chars     MiniLM-L6-v2    Vector store
                                                      |
[Input User] -> [Intent Detect] -> [RAG Retrieve] -> [Prompt] -> [LLM (Groq)] -> [PRD Output]
 "Buat PRD..."   generate/qna      Cari relevan    Konteks          8B cloud    PRD jadi
```

**Komponen:**
- Data Ingestion: ekstrak teks PDF, chunking, embedding, ChromaDB
- Retrieval: semantic search (RAG)
- Intent Detection: bedakan pertanyaan vs generate
- LLM: Groq cloud (`llama-3.1-8b-instant`)
- Evaluasi: ROUGE-1/2/L

Pipeline ini identik dengan yang berjalan di backend FastAPI (`App/backend/main.py`) dan frontend React (`App/frontend/`).

In [ ]:
# Cleanup
if config.TEMP_DRIVE_DIR.exists():
    shutil.rmtree(config.TEMP_DRIVE_DIR)
    print('Temporary Drive files dihapus.')
print('Pipeline selesai.')
